Tools: 
Models can request to call tools that perform tasks such as 
- fetching data from a database, 
- searching the web, or 
- running the code
Tools are pairing of: 
1. A schema, including the name of the tool, a description, and/or arguments definition (often a JSON format)
2. A function or coroutine to execute


How to create a tool?

1. Firstly invoke model 

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("give me 30 characters short answer, why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Question:** "why do parrots talk?"\n   - **Constraint:** "30 characters short answer"\n   - **Key Requirement:** The answer must be exactly or at most 30 characters (including spaces/punctuation), and it should address why parrots talk.\n\n2.  **Identify Core Scientific Reason:**\n   - Parrots mimic sounds to communicate, bond socially, and attract attention/mates.\n   - Key concepts: social bonding, communication, mimicry, survival.\n\n3.  **Brainstorm Short Answers (aiming for ≤30 chars):**\n   - To bond and communicate. (24)\n   - Social bonding & mimicry. (25)\n   - To connect with their flock. (27)\n   - Mimicry for social bonds. (25)\n   - Social bonding instinct. (23)\n   - To bond and attract mates. (26)\n   - Instinctual social mimicry. (27)\n   - For social bonding & play. (26)\n   - To communicate & bond. (22)\n   - Social connection instinct. (26)\n\n4.  **Check Character Counts

Now jump to, how to create a tool?

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the location's weather.""" # docstring - will tell the model what this tool does
    return f"The weather in {location} is sunny."

model_with_tool = model.bind_tools([get_weather])


Binding Tool to model is done, Now how do we call it?

In [ ]:
response = model_with_tool.invoke("what is the weather in New York?")
response.tool_calls
print(response)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: "what is the weather in New York?"\n2.  Identify the required tool: `get_weather`\n3.  Identify the required parameter: `location` = "New York"\n4.  Call the tool.\n5.  Format the response based on the tool\'s output.\n\nLet\'s call the tool. \n`get_weather(location="New York")`\nWait, should I specify "New York City" or just "New York"? The prompt says "New York", I\'ll stick to that.\nProceed. \nTool call: `get_weather(location="New York")`\nOutput handling: I will wait for the result, but since I\'m simulating the thought process, I\'ll just make the call.\nActually, I am generating the tool call now.\nDone. \nProceeding. \n`print(get_weather(location="New York"))` -> wait, I just output the tool call block.\nAll good. \nLet\'s generate. \n[Self-Correction/Verification during thought]\nParameter matches schema: `location` (string) -> "New York". Required: yes.\nTool name: `get_we

### Tool Execution Loop 

In [7]:
### Tool Execution Loops
messages = [{"role": "user", "content": "What's is the weather in Boston?"}]
ai_msgs = model_with_tool.invoke(messages)
print(ai_msgs)
messages.append(ai_msgs)
print(messages)

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify user intent: User wants to know the weather in Boston.\n2.  Identify required tool: `get_weather`\n3.  Check parameters: `location` (string) is required. User provided "Boston".\n4.  Call tool: `get_weather(location="Boston")`\n5.  Return result to user.✅\n', 'tool_calls': [{'id': 'ycb8p2v3s', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 275, 'total_tokens': 382, 'completion_time': 0.208945441, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.02413672, 'prompt_tokens_details': None, 'queue_time': 0.181936585, 'total_time': 0.233082161}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_56b71ffadd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0428d-1015-77e3-accb-8c92fd2